In [ ]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

# R0计算设置
R0_TARGET_AFTER_PAUSE_SEC = 0.5   # pause段最后一个测量点之后外推的R0时间
R0_FIT_POINT_START = 2            # 默认使用有效pulse第2-6点
R0_FIT_POINT_END = 6              # 拟合终点固定为有效pulse第6点
FIRST_TWO_VOLTAGE_EQUAL_ATOL = 1e-9  # 前两点电压视为相同的绝对容差(V)
ZERO_CURRENT_LIMIT = 1e-6
VOLTAGE_JUMP_LIMIT = 1e-3
MAX_PAUSE_TO_PULSE_GAP_SEC = 5.0  # pause末点到pulse首点严格大于5s则剔除
R0_EARLY_WINDOW_FLAG = "后段电流不稳定，R0仅使用早期稳定窗口"

# R0置信度规则：
# “干净标签”的有效R0可按权重1.0使用。
TRUSTED_R0_QUALITY_FLAGS = {
    "正常",
    R0_EARLY_WINDOW_FLAG,
    "首点0且电压跳变"
}

R0_CONFIDENCE_FULL = "完全可信（权重1.0）"
R0_CONFIDENCE_REVIEW = "需复核（权重0.5）"
R0_CONFIDENCE_INVALID = "不可用（权重0.0）"

TIME_DIFF_OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "ID",
    "Zustand/Current"
]

# R0及质量检验列
PULSE_OUTPUT_COLUMNS = TIME_DIFF_OUTPUT_COLUMNS + [
    "R0",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence"
]


In [4]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [5]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间、电流、电压处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")
    df_td["Voltage"] = pd.to_numeric(df_td["Voltage"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (
        df_td.groupby("File")["Time"].diff() / pd.Timedelta(hours=1)
    )

    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (
        df_td["Time"] - cycle_start_time
    ) / pd.Timedelta(minutes=1)

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 生成 Zustand/Current，并保留完整 time_diff_sequence
    # -----------------------------
    df_td["Zustand/Current"] = (
        df_td["Zustand"]
        + "/"
        + df_td["Current"].astype(float).round(1).astype(str)
    )

    # 这个表保留所有状态点，后面用于筛选 PAUO
    time_diff_sequence = df_td[TIME_DIFF_OUTPUT_COLUMNS].copy()

    # 只保留 CHA / DCH 作为 pulse_sequence
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def get_effective_start_pos(group):
    # 返回pulse段内第一个非零有效电流点的位置。
        current_abs = group["Current"].abs().to_numpy()
        non_zero_pos = np.flatnonzero(current_abs > ZERO_CURRENT_LIMIT)

        if len(non_zero_pos) == 0:
            return None

        return int(non_zero_pos[0])

    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            return True

        # 如果首个测量点 Current == 0，则从第一个非零有效点开始判断稳定性
        current_values = group["Current"].iloc[effective_start_pos:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True

    # 不再因为pulse后段电流下降而提前删除整段数据。
    # R0是否可计算，改为在下面仅依据实际拟合窗口内的电流稳定性判断。

    # -----------------------------
    # 7. 计算R0，并且每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def add_quality(base_quality, new_quality):
        if base_quality == "正常":
            return new_quality
        return base_quality + "；" + new_quality

    def calculate_r0_for_segment(group, effective_start_pos):
        group = group.sort_values("Time")

        r0_result = {
            "R0": np.nan,
            "R0_Target_Time": pd.NaT,
            "Pause_to_Pulse_Time_Diff_s": np.nan,
            "R0_Quality": "正常"
        }

        file_name = group["File"].iloc[0]
        pulse_start_time = group["Time"].iloc[0]
        first_current = group["Current"].iloc[0]

        previous_pause = df_td[
            (df_td["File"] == file_name)
            & (df_td["Time"] < pulse_start_time)
            & (df_td["Zustand"].str.startswith("PAU", na=False))
        ].sort_values("Time").tail(1)

        if previous_pause.empty:
            r0_result["R0_Quality"] = "无法计算R0：无前置pause点"
            return r0_result

        pause_time = previous_pause["Time"].iloc[0]
        pause_voltage = previous_pause["Voltage"].iloc[0]

        pause_to_pulse_gap_s = (pulse_start_time - pause_time).total_seconds()
        r0_result["Pause_to_Pulse_Time_Diff_s"] = pause_to_pulse_gap_s

        target_time = pause_time + pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC)
        r0_result["R0_Target_Time"] = target_time

        # 大于5s：标记为无效，不再进行长距离反向外推
        if pause_to_pulse_gap_s > MAX_PAUSE_TO_PULSE_GAP_SEC:
            r0_result["R0_Quality"] = "pause结束点到pulse首点时间差>5s"
            return r0_result

        # pulse段第一个点为0时：无论voltage是否跳变，R0计算都从第一个非零有效点开始；
        # 若voltage已经跳变，则额外给质量标记。
        if abs(first_current) <= ZERO_CURRENT_LIMIT:
            first_voltage = group["Voltage"].iloc[0]

            if (
                pd.notna(first_voltage)
                and pd.notna(pause_voltage)
                and abs(first_voltage - pause_voltage) > VOLTAGE_JUMP_LIMIT
            ):
                r0_result["R0_Quality"] = "首点0且电压跳变"

        # 默认使用有效pulse第2-6点进行线性拟合。
        # 若有效pulse第1点与第2点的电压相同（可能是重复采样），
        # 则跳过前两点，改用第3-6点进行线性外推。
        fit_point_start = R0_FIT_POINT_START

        if effective_start_pos + 1 < len(group):
            first_pulse_voltage = group["Voltage"].iloc[effective_start_pos]
            second_pulse_voltage = group["Voltage"].iloc[effective_start_pos + 1]

            first_two_voltage_equal = (
                pd.notna(first_pulse_voltage)
                and pd.notna(second_pulse_voltage)
                and np.isclose(
                    float(first_pulse_voltage),
                    float(second_pulse_voltage),
                    rtol=0.0,
                    atol=FIRST_TWO_VOLTAGE_EQUAL_ATOL
                )
            )

            if first_two_voltage_equal:
                fit_point_start = 3

        fit_start_pos = effective_start_pos + fit_point_start - 1
        fit_end_pos = effective_start_pos + R0_FIT_POINT_END

        fit_points = group.iloc[fit_start_pos:fit_end_pos].dropna(
            subset=["Time", "Voltage", "Current"]
        )

        if len(fit_points) < 2:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：拟合点不足"
            )
            return r0_result

        # R0只要求实际参与2-6点（或3-6点）拟合的早期窗口电流稳定。
        fit_current_values = fit_points["Current"]
        fit_current_abs_level = round(fit_current_values.abs().iloc[0], 1)
        fit_current_std = fit_current_values.std()

        if fit_current_abs_level == 1.5:
            fit_current_unstable = fit_current_std > STD_LIMIT_1P5A
        elif fit_current_abs_level == 3.0:
            fit_current_unstable = fit_current_std > STD_LIMIT_3A
        else:
            fit_current_unstable = True

        if fit_current_unstable:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：早期拟合窗口电流不稳定"
            )
            return r0_result

        # 若整段电流不稳定、但早期拟合窗口稳定，仍计算R0并添加专用flag。
        if is_bad_current_segment(group):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                R0_EARLY_WINDOW_FLAG
            )

        effective_current = group["Current"].iloc[effective_start_pos]

        if abs(effective_current) <= ZERO_CURRENT_LIMIT:
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：无非零有效电流"
            )
            return r0_result

        if pd.isna(pause_voltage):
            r0_result["R0_Quality"] = add_quality(
                r0_result["R0_Quality"],
                "无法计算R0：pause电压缺失"
            )
            return r0_result
        
        # 判断采样点距离pulse起点的距离
        x_sec = (fit_points["Time"] - target_time) / pd.Timedelta(seconds=1)

        y_voltage = fit_points["Voltage"].astype(float)

        slope, intercept = np.polyfit(x_sec.to_numpy(), y_voltage.to_numpy(), 1)
        extrapolated_voltage = intercept

        r0_result["R0"] = abs(
            (extrapolated_voltage - pause_voltage) / effective_current
        )

        return r0_result

    selected_indices = []
    r0_results = {}

    for _, group in pulse_sequence.groupby(["File", "pulse_segment_id"], sort=False):
        group = group.sort_values("Time")

        effective_start_pos = get_effective_start_pos(group)
        if effective_start_pos is None:
            continue

        # 保留原逻辑：每段用“有效pulse起点后的第2个测量点”记录；
        # 如果点数不够，则退回到有效pulse起点本身。
        record_pos = effective_start_pos + 1
        if record_pos >= len(group):
            record_pos = effective_start_pos

        record_index = group.index[record_pos]
        selected_indices.append(record_index)

        r0_results[record_index] = calculate_r0_for_segment(
            group,
            effective_start_pos
        )

    pulse_sequence = pulse_sequence.loc[selected_indices].copy()

    r0_result_columns = [
        "R0",
        "R0_Target_Time",
        "Pause_to_Pulse_Time_Diff_s",
        "R0_Quality"
    ]

    for column in r0_result_columns:
        pulse_sequence[column] = pulse_sequence.index.map(
            lambda idx, col=column: r0_results[idx][col]
        )

    pulse_sequence = pulse_sequence.reset_index(drop=True)

    # -----------------------------
    # 8. 生成逐条R0置信度
    # -----------------------------
    def classify_r0_confidence(row):
        r0_value = row["R0"]
        quality_text = str(row["R0_Quality"]).strip()

        # 没有有效R0时，无论带有什么flag，都不能参与R0分析。
        if pd.isna(r0_value) or not np.isfinite(r0_value):
            return R0_CONFIDENCE_INVALID

        flags = {
            flag.strip()
            for flag in quality_text.split("；")
            if flag.strip()
        }

        # 只要一条记录所含flag全部属于已验证的干净标签，就给权重1.0。
        if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
            return R0_CONFIDENCE_FULL

        # 将来若新增了仍可计算R0、但尚未验证的flag，先标记为需复核。
        return R0_CONFIDENCE_REVIEW

    pulse_sequence["R0_Confidence"] = pulse_sequence.apply(
        classify_r0_confidence,
        axis=1
    )

    # -----------------------------
    # 9. 只保留最终输出列
    # -----------------------------
    pulse_sequence = pulse_sequence[PULSE_OUTPUT_COLUMNS].copy()

    return pulse_sequence, time_diff_sequence

pulse_sequence, time_diff_sequence = build_time_diff_sequence(df)

# 中间结果预览；统一的R0_Quality统计放在最终绘图cell中，避免重复输出。
display(pulse_sequence)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.760000+00:00,-1.499470,4.036492,DCH,12_17,DCH/-1.5,NaN,NaT,NaN,无法计算R0：无前置pause点,不可用（权重0.0）
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,0.025070,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,0.024941,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.310000+00:00,2.995714,4.165722,CHA,12_29,CHA/3.0,0.024550,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.640000+00:00,-1.498840,3.725950,DCH,12_35,DCH/-1.5,NaN,2024-11-12 15:36:43.330000+00:00,15553.57,pause结束点到pulse首点时间差>5s,不可用（权重0.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,0.017858,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
280,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.810000+00:00,-1.497851,3.192480,DCH,9_54,DCH/-1.5,NaN,2024-10-24 00:01:13.750000+00:00,15695.30,pause结束点到pulse首点时间差>5s,不可用（权重0.0）
281,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,0.019617,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）
282,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,0.019862,2024-10-24 06:24:14.600000+00:00,0.86,正常,完全可信（权重1.0）


In [6]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
# -----------------------------
def filter_pulse(df):

    pulse_sequence_filter = df[~df["Zustand/Current"].isin(["DCH/-1.5"])].copy()
    return pulse_sequence_filter

filtered_pulse = filter_pulse(pulse_sequence)
display(filtered_pulse)


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.740000+00:00,1.499155,4.127363,CHA,12_21,CHA/1.5,0.025070,2024-11-12 13:04:35.250000+00:00,0.76,正常,完全可信（权重1.0）
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.910000+00:00,-2.998064,4.009585,DCH,12_25,DCH/-3.0,0.024941,2024-11-12 14:05:18.390000+00:00,0.81,正常,完全可信（权重1.0）
3,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 15:06:22.310000+00:00,2.995714,4.165722,CHA,12_29,CHA/3.0,0.024550,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.940000+00:00,1.499604,3.805598,CHA,12_39,CHA/1.5,0.018330,2024-11-12 20:56:39.410000+00:00,0.81,正常,完全可信（权重1.0）
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.250000+00:00,-2.999144,3.719279,DCH,12_43,DCH/-3.0,0.018422,2024-11-12 21:57:22.670000+00:00,0.85,正常,完全可信（权重1.0）
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.530000+00:00,-2.999504,3.717166,DCH,9_44,DCH/-3.0,0.017919,2024-10-23 22:29:48.980000+00:00,0.84,正常,完全可信（权重1.0）
279,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.820000+00:00,2.999671,3.831503,CHA,9_48,CHA/3.0,0.017858,2024-10-23 23:30:52.300000+00:00,0.81,正常,完全可信（权重1.0）
281,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:32.010000+00:00,1.498525,3.332511,CHA,9_58,CHA/1.5,0.019617,2024-10-24 05:23:31.420000+00:00,0.88,正常,完全可信（权重1.0）
282,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:15.220000+00:00,-2.996265,3.241513,DCH,9_62,DCH/-3.0,0.019862,2024-10-24 06:24:14.600000+00:00,0.86,正常,完全可信（权重1.0）


In [7]:
# =============================================================================
# 10. R0结果按flag输出 + 按SOC / pulse / current分类绘图
# =============================================================================
# 说明：
#   1. R0 本身已经在 build_time_diff_sequence(df) 中计算完成；
#   2. 这里基于 filtered_pulse 输出 R0 结果和 flag 统计；
#   3. R0 单位从 Ohm 转为 mOhm；
#   4. 绘图分类方式参考：
#      10% / 50% / 90% SOC 分成三个子图；
#      每条线按 SOC + CHA/DCH + 电流大小 分类。

# -----------------------------
# 1. 选择R0结果来源
# -----------------------------
if "filtered_pulse" in globals():
    r0_source = filtered_pulse.copy()
elif "pulse_sequence" in globals():
    r0_source = pulse_sequence.copy()
else:
    raise NameError("请先运行前面的cell，生成 pulse_sequence 或 filtered_pulse。")

required_columns = [
    "SOH",
    "SOC",
    "Time",
    "Current",
    "Voltage",
    "Zustand",
    "Zustand/Current",
    "R0",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality"
]

missing_columns = [col for col in required_columns if col not in r0_source.columns]
if missing_columns:
    raise KeyError(f"R0结果缺少必要列: {missing_columns}")

# -----------------------------
# 2. 整理R0结果
# -----------------------------
r0_result = r0_source.copy()

r0_result["SOH"] = pd.to_numeric(r0_result["SOH"], errors="coerce")
r0_result["Current"] = pd.to_numeric(r0_result["Current"], errors="coerce")
r0_result["R0"] = pd.to_numeric(r0_result["R0"], errors="coerce")

# R0原单位为 Ohm，这里转换为 mOhm，方便和图里的量级一致
r0_result["R0_mOhm"] = r0_result["R0"] * 1000

r0_result["R0_Quality"] = (
    r0_result["R0_Quality"]
    .fillna("缺失")
    .astype(str)
    .str.strip()
)

r0_result["Current_abs_A"] = r0_result["Current"].abs().round(1)

r0_result["Current_Label"] = r0_result["Current_abs_A"].map(
    lambda x: f"{x:.1f}A" if pd.notna(x) else "UnknownA"
)

r0_result["SOC_pulse_current"] = (
    r0_result["SOC"].astype(str)
    + " "
    + r0_result["Zustand"].astype(str)
    + " "
    + r0_result["Current_Label"]
)

r0_result["R0_Is_Valid"] = (
    r0_result["R0_mOhm"].notna()
    & np.isfinite(r0_result["R0_mOhm"])
)

# 逐条记录的R0置信度。即使前面cell尚未重跑，这里也会重新生成。
def classify_r0_confidence(row):
    if not row["R0_Is_Valid"]:
        return R0_CONFIDENCE_INVALID

    flags = {
        flag.strip()
        for flag in str(row["R0_Quality"]).split("；")
        if flag.strip()
    }

    if flags and flags.issubset(TRUSTED_R0_QUALITY_FLAGS):
        return R0_CONFIDENCE_FULL

    return R0_CONFIDENCE_REVIEW

r0_result["R0_Confidence"] = r0_result.apply(
    classify_r0_confidence,
    axis=1
)

result_display_columns = [
    "SOH",
    "SOC",
    "Time",
    "Zustand",
    "Current",
    "Voltage",
    "R0",
    "R0_mOhm",
    "R0_Target_Time",
    "Pause_to_Pulse_Time_Diff_s",
    "R0_Quality",
    "R0_Confidence",
    "SOC_pulse_current",
    "File"
]

print("R0结果：按 R0_Quality / SOC / SOH 排序")
display(
    r0_result[result_display_columns]
    .sort_values(
        ["R0_Quality", "SOC", "SOH", "Zustand", "Current"],
        ascending=[True, True, False, True, True]
    )
    .reset_index(drop=True)
)

# -----------------------------
# 3. R0_Quality统计
# -----------------------------
# 若一条记录包含多个以“；”分隔的flag，会拆开后分别统计。
r0_quality_summary = (
    r0_result
    .assign(R0_Quality_Flag=r0_result["R0_Quality"].str.split("；"))
    .explode("R0_Quality_Flag")
)

r0_quality_summary["R0_Quality_Flag"] = (
    r0_quality_summary["R0_Quality_Flag"]
    .fillna("缺失")
    .astype(str)
    .str.strip()
)

r0_quality_summary = (
    r0_quality_summary
    .groupby("R0_Quality_Flag", dropna=False)
    .agg(
        Flag_Count=("R0_Quality_Flag", "size"),
        Valid_R0_Count=("R0_Is_Valid", "sum")
    )
    .reset_index()
    .sort_values("Flag_Count", ascending=False)
)

r0_quality_summary["Percent_of_segments"] = (
    r0_quality_summary["Flag_Count"] / len(r0_result) * 100
)

# flag级别的建议置信度：
# 3（正常）、1（后段不稳但早期窗口稳定）、4（首点0且电压跳变）
# 都作为干净标签，建议权重1.0。
r0_quality_summary["R0_Confidence"] = np.select(
    [
        (
            r0_quality_summary["R0_Quality_Flag"].isin(
                TRUSTED_R0_QUALITY_FLAGS
            )
            & r0_quality_summary["Valid_R0_Count"].gt(0)
        ),
        r0_quality_summary["Valid_R0_Count"].eq(0)
    ],
    [
        R0_CONFIDENCE_FULL,
        R0_CONFIDENCE_INVALID
    ],
    default=R0_CONFIDENCE_REVIEW
)

print("R0_Quality统计：")
display(r0_quality_summary)

# -----------------------------
# 4. 按图示方式绘图：SOH vs R0，分SOC子图，按 SOC / pulse / current 分线
# -----------------------------
# 绘制正常R0，以及“后段电流不稳定但早期拟合窗口稳定”的有效R0
r0_plot_data = r0_result[
    r0_result["R0_Is_Valid"]
    & r0_result["R0_Quality"].isin([
        "正常",
        R0_EARLY_WINDOW_FLAG
    ])
].copy()

if r0_plot_data.empty:
    print("没有可绘制的有效R0数据。")
else:
    soc_plot_order = ["10%", "50%", "90%"]
    available_soc_order = [
        soc for soc in soc_plot_order
        if soc in r0_plot_data["SOC"].astype(str).unique()
    ]

    # 如果出现了不在默认顺序中的SOC，也保留在后面
    extra_soc = [
        soc for soc in r0_plot_data["SOC"].astype(str).unique()
        if soc not in available_soc_order
    ]

    available_soc_order = available_soc_order + sorted(extra_soc)

    fig = make_subplots(
        rows=1,
        cols=len(available_soc_order),
        subplot_titles=[f"{soc} SOC" for soc in available_soc_order],
        shared_yaxes=False,
        horizontal_spacing=0.08
    )

    shown_legend = set()

    for col_idx, soc in enumerate(available_soc_order, start=1):
        soc_data = r0_plot_data[
            r0_plot_data["SOC"].astype(str) == soc
        ].copy()

        # 按SOH从高到低排序，配合反向x轴，视觉上和示例图一致
        soc_data = soc_data.sort_values(
            ["SOC_pulse_current", "SOH"],
            ascending=[True, False]
        )

        for label, group in soc_data.groupby("SOC_pulse_current", sort=True):
            group = group.sort_values("SOH", ascending=False)

            fig.add_trace(
                go.Scatter(
                    x=group["SOH"],
                    y=group["R0_mOhm"],
                    mode="lines+markers",
                    name=label,
                    legendgroup=label,
                    showlegend=label not in shown_legend
                ),
                row=1,
                col=col_idx
            )

            shown_legend.add(label)

        fig.update_xaxes(
            title_text="SOH (%)",
            autorange="reversed",
            row=1,
            col=col_idx
        )

        fig.update_yaxes(
            title_text="R0 (mOhm)",
            row=1,
            col=col_idx
        )

    fig.update_layout(
        title="SOH vs SOC 10%, 50%, 90% at R0",
        legend_title_text="SOC / pulse / current",
        width=1500,
        height=600,
        template="plotly_white"
    )

    fig.show()


R0结果：按 R0_Quality / SOC / SOH 排序


,SOH,SOC,Time,Zustand,Current,Voltage,R0,R0_mOhm,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence,SOC_pulse_current,File
0,81.7,50%,2025-06-26 20:02:05.060000+00:00,CHA,2.999851,3.879424,NaN,NaN,2025-06-26 20:01:53.770000+00:00,11.66,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,50% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
1,81.7,90%,2025-06-26 12:20:22.870000+00:00,CHA,2.370311,4.199966,NaN,NaN,2025-06-26 12:20:04.430000+00:00,18.80,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,90% CHA 2.4A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
2,90.3,90%,2024-11-12 15:06:22.310000+00:00,CHA,2.995714,4.165722,0.024550,24.550339,2024-11-12 15:06:21.920000+00:00,0.79,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...
3,89.1,90%,2024-12-01 15:26:19.060000+00:00,CHA,2.997153,4.174616,0.026980,26.979597,2024-12-01 15:26:18.550000+00:00,0.80,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM14_...
4,88.0,90%,2024-12-19 01:29:21.090000+00:00,CHA,2.995175,4.176729,0.027769,27.769312,2024-12-19 01:29:20.630000+00:00,0.80,后段电流不稳定，R0仅使用早期稳定窗口,完全可信（权重1.0）,90% CHA 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM16_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,78.6,90%,2025-11-09 20:12:28.080000+00:00,CHA,1.493757,4.145041,0.032455,32.455231,2025-11-09 20:12:27.730000+00:00,0.75,首点0且电压跳变,完全可信（权重1.0）,90% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
212,78.6,90%,2025-11-09 21:13:10.940000+00:00,DCH,-2.991047,3.982345,0.032661,32.660834,2025-11-09 21:13:10.630000+00:00,0.74,首点0且电压跳变,完全可信（权重1.0）,90% DCH 3.0A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
213,77.8,90%,2025-12-14 13:48:16.550000+00:00,CHA,1.496726,4.151712,0.039542,39.542019,2025-12-14 13:48:16.140000+00:00,0.76,首点0且电压跳变,完全可信（权重1.0）,90% CHA 1.5A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM52_...
214,79.4,90%,2025-10-04 17:07:30.340000+00:00,CHA,2.597305,4.200300,NaN,NaN,2025-10-04 17:07:29.990000+00:00,0.75,首点0且电压跳变；无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,90% CHA 2.6A,METABatt_Sony_Murata_18650VTC6_007_pulse_BM44_...


R0_Quality统计：


,R0_Quality_Flag,Flag_Count,Valid_R0_Count,Percent_of_segments,R0_Confidence
3,正常,179,179,82.870370,完全可信（权重1.0）
4,首点0且电压跳变,16,14,7.407407,完全可信（权重1.0）
2,无法计算R0：早期拟合窗口电流不稳定,11,0,5.092593,不可用（权重0.0）
1,后段电流不稳定，R0仅使用早期稳定窗口,10,10,4.629630,完全可信（权重1.0）
0,pause结束点到pulse首点时间差>5s,2,0,0.925926,不可用（权重0.0）


In [ ]:
# =============================================================================
# 12. Randles + finite-length Warburg Open：核心函数
# =============================================================================
# 电路拓扑：
#   R0 + (R1 || C1) + W_open
#
# Warburg Open（反射/阻塞边界）：
#   Z_W(s) = Rd * coth(sqrt(s*td)) / sqrt(s*td)
#
# 使用恒等式：
#   coth(z)/z = 1/z^2 + 2 * sum_{n=1..inf} 1/(z^2 + n^2*pi^2)
#
# 因而可写成一个积分状态 + N 个一阶指数状态：
#   dVw0/dt = (Rd/td) * I
#   dVwn/dt = -(n^2*pi^2/td)*Vwn + (2*Rd/td)*I
#
# 截断到 N=3~5 后，每个扩散模态仍是标准指数响应，可直接使用
# scipy.optimize.least_squares 拟合，不需要分数阶数值算法。
#
# 重要单位：
#   r0_result["R0"] 的原始单位是 Ohm；
#   r0_result["R0_mOhm"] 的单位是 mOhm。
# =============================================================================

WARBURG_MAIN_TERMS = 5
WARBURG_CHECK_TERMS = 3
WARBURG_NOISE_FLOOR_MV = 0.5   # 可按设备实际电压噪声修改
RUN_LEGACY_RC2_COMPARISON = True

# Stage 2 联合拟合中，pulse 与后续 PAUO 的总权重。
# 两段都完整纳入；权重按各段的时间支撑归一化。
PULSE_WEIGHT_TOTAL = 0.55
PAUO_WEIGHT_TOTAL = 0.45

# Warburg 初始值。R1_fraction 表示总动态电阻初值中分配给 R1 的比例。
RANDLES_WARBURG_INITIAL_BY_SOC = {
    "90%": {"R1_fraction": 0.40, "tau1": 2.0, "td": 150.0},
    "50%": {"R1_fraction": 0.35, "tau1": 4.0, "td": 300.0},
}

# 10% SOC 不再使用 Warburg 结构。
# 依据（原始逐点数据诊断）：
#   1) 对 PAUO 段做自由双指数拟合可达噪声底（RMSE 0.15~0.32 mV），
#      真实时间常数为 快~15-26s、慢~340-1030s，且两者之间无显著弛豫成分；
#   2) 有限长 Warburg 的模态阶梯（td, td/4, td/9, ...，幅值比固定）
#      与该"两个离散指数"的谱结构不相容，任何 td 都会注入数据中不存在的
#      中间模态，导致 tau1/td 角色互换、Rd-td 沿比值方向跑飞等伪解；
#   3) 10% SOC 处 OCV 曲线陡峭，脉冲期间 OCV 随电荷持续移动，
#      需要一个库仑积分项 k*Q(t) 来承接（这正是此前 Warburg 积分项
#      被优化器"滥用"去拟合的成分）。
# 因此 10% SOC 改用：R0 + R1||C1 + R2||C2 + k*Q(t)，见 fit_rc2_coulomb_10pct。

# 采用宽而统一的正值下界，避免 1.5A CHA 等工况被人为卡在非零下界。
RANDLES_WARBURG_BOUNDS_BY_SOC = {
    "90%": {
        "R1_max": 0.10, "Rd_max": 0.25,
        "tau1_min": 0.20, "tau1_max": 40.0,
        "td_min": 5.0, "td_max": 3000.0,
    },
    "50%": {
        "R1_max": 0.15, "Rd_max": 0.35,
        "tau1_min": 0.20, "tau1_max": 60.0,
        "td_min": 5.0, "td_max": 6000.0,
    },
}

# 10% SOC 的 RC2+Coulomb 模型配置（数值已用全部 72 段 10% SOC
# 原始数据验证：内点率 92~100%，RMSE 0.5~0.7 mV，tau1≈10-16s，
# DCH 慢过程 R2≈208 mOhm / tau2≈844 s 均为内点解）。
# 仅 tau2 初值与 R2 上界按子工况区分，其余共用。
RC2_COULOMB_10PCT_CONFIG = {
    "DCH_3A":   {"tau2_init": 1000.0, "R2_max": 0.80},
    "CHA_3A":   {"tau2_init": 450.0,  "R2_max": 0.30},
    "CHA_1.5A": {"tau2_init": 300.0,  "R2_max": 0.30},
}
RC2_COULOMB_10PCT_SHARED = {
    "R1_init": 0.015, "R2_init": 0.05, "tau1_init": 18.0,
    "k_init": 1e-4, "R1_max": 0.10,
    "tau1_min": 0.5, "tau1_max": 60.0,
    "tau2_min": 30.0, "tau2_max": 1600.0,
    "k_min": 0.0, "k_max": 1.5e-3,
    "ocv_offset_slack_v": 0.010,
}


def _is_pulse_state(value):
    return str(value).strip().startswith(("CHA", "DCH"))


def _r0_ohm_from_row(row):
    """统一把 R0 转为 Ohm，优先使用显式的 R0_mOhm 列。"""
    r0_mohm = pd.to_numeric(row.get("R0_mOhm", np.nan), errors="coerce")
    if pd.notna(r0_mohm) and np.isfinite(r0_mohm):
        return float(r0_mohm) / 1000.0

    r0_ohm = pd.to_numeric(row.get("R0", np.nan), errors="coerce")
    if pd.notna(r0_ohm) and np.isfinite(r0_ohm):
        return float(r0_ohm)

    return np.nan


def _time_support_weights(t_s, mask, total_weight):
    """
    按每个采样点代表的时间跨度分配权重，并把该片段总平方权重
    归一化为 total_weight。这样 pulse 与长 PAUO 尾段都能稳定参与拟合。
    """
    t_s = np.asarray(t_s, dtype=float)
    mask = np.asarray(mask, dtype=bool)
    support = np.zeros_like(t_s, dtype=float)

    if len(t_s) == 0 or not mask.any():
        return support

    if len(t_s) == 1:
        support[0] = 1.0
    else:
        dt = np.diff(t_s)
        positive_dt = dt[dt > 0]
        fallback = float(np.median(positive_dt)) if len(positive_dt) else 1.0

        support[0] = dt[0] if dt[0] > 0 else fallback
        support[-1] = dt[-1] if dt[-1] > 0 else fallback
        if len(t_s) > 2:
            support[1:-1] = 0.5 * (
                np.maximum(dt[:-1], 0.0) + np.maximum(dt[1:], 0.0)
            )

        support[support <= 0] = fallback

        positive_support = support[support > 0]
        cap = np.quantile(positive_support, 0.95) if len(positive_support) else fallback
        support = np.minimum(support, max(float(cap), fallback))

    segment_support = support[mask]
    denom = float(segment_support.sum())

    weights = np.zeros_like(t_s, dtype=float)
    if denom > 0:
        weights[mask] = np.sqrt(total_weight * segment_support / denom)

    return weights


def build_stage2_fit_windows(time_diff_sequence, r0_result):
    """
    构造 Stage 2 拟合窗口：
      - 前一个 PAUO 末端点只用于确定初始 OCV；
      - 拟合数据从 pulse 首个有效点开始；
      - 当前 pulse 全段 + 紧随其后的完整 PAUO 全段联合拟合。
    """
    seq = time_diff_sequence.copy()
    seq["Zustand_clean"] = (
        seq["Zustand"]
        .astype(str)
        .str.strip()
        .str.replace(r"\*+$", "", regex=True)
    )
    seq["Time_dt"] = pd.to_datetime(seq["Time"], utc=True, errors="coerce")
    seq["Current"] = pd.to_numeric(seq["Current"], errors="coerce")
    seq["Voltage"] = pd.to_numeric(seq["Voltage"], errors="coerce")

    seq = (
        seq
        .dropna(subset=["File", "Time_dt", "Current", "Voltage"])
        .sort_values(["File", "Time_dt"])
        .reset_index(drop=True)
    )

    seq["segment_id"] = (
        seq["File"].ne(seq["File"].shift())
        | seq["Zustand_clean"].ne(seq["Zustand_clean"].shift())
    ).cumsum()

    seq_id_parts = seq["ID"].astype(str).str.extract(r"^(.*)_(\d+)$")
    seq["id_prefix"] = seq_id_parts[0]
    seq["id_number"] = pd.to_numeric(seq_id_parts[1], errors="coerce")

    r0_table = r0_result.copy()
    r0_table["Time_dt"] = pd.to_datetime(
        r0_table["Time"], utc=True, errors="coerce"
    )

    windows = []
    skipped_rows = []

    for _, row in r0_table.iterrows():
        file_name = row.get("File")
        pulse_time = row.get("Time_dt")
        r0_ohm = _r0_ohm_from_row(row)

        skip_reason = None

        if pd.isna(pulse_time):
            skip_reason = "R0记录时间无效"
        elif not np.isfinite(r0_ohm):
            skip_reason = "R0无效"

        file_seq = seq[seq["File"].eq(file_name)].copy()
        if skip_reason is None and file_seq.empty:
            skip_reason = "找不到对应文件数据"

        if skip_reason is not None:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = skip_reason
            skipped_rows.append(skipped)
            continue

        pulse_candidates = file_seq[
            file_seq["Time_dt"].eq(pulse_time)
            & file_seq["Zustand"].map(_is_pulse_state)
        ]

        if pulse_candidates.empty:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "找不到R0记录对应的pulse片段"
            skipped_rows.append(skipped)
            continue

        pulse_segment_id = pulse_candidates.iloc[0]["segment_id"]
        pulse_df = file_seq[
            file_seq["segment_id"].eq(pulse_segment_id)
            & file_seq["Zustand"].map(_is_pulse_state)
        ].copy()
        pulse_df = pulse_df.sort_values("Time_dt").reset_index(drop=True)

        while (
            len(pulse_df) >= 2
            and abs(float(pulse_df["Current"].iloc[0])) <= ZERO_CURRENT_LIMIT
        ):
            pulse_df = pulse_df.iloc[1:].reset_index(drop=True)

        if len(pulse_df) < 5:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "pulse有效点少于5个"
            skipped_rows.append(skipped)
            continue

        pulse_start_time = pulse_df["Time_dt"].iloc[0]
        pulse_end_time = pulse_df["Time_dt"].iloc[-1]

        prev_pauo = file_seq[
            file_seq["Time_dt"].lt(pulse_start_time)
            & file_seq["Zustand_clean"].eq("PAUO")
        ].copy()

        if prev_pauo.empty:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "无前置PAUO点"
            skipped_rows.append(skipped)
            continue

        prev_pauo_point = prev_pauo.sort_values("Time_dt").iloc[-1]

        id_parts = str(row.get("ID", "")).rsplit("_", 1)
        if len(id_parts) != 2:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "ID无法解析"
            skipped_rows.append(skipped)
            continue

        pulse_id_prefix = id_parts[0]
        pulse_id_number = pd.to_numeric(id_parts[1], errors="coerce")
        if pd.isna(pulse_id_number):
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "ID序号无法解析"
            skipped_rows.append(skipped)
            continue

        next_pauo_df = file_seq[
            file_seq["Time_dt"].gt(pulse_end_time)
            & file_seq["Zustand_clean"].eq("PAUO")
            & file_seq["id_prefix"].eq(pulse_id_prefix)
            & file_seq["id_number"].eq(pulse_id_number + 1)
        ].copy()
        next_pauo_df = next_pauo_df.sort_values("Time_dt").reset_index(drop=True)

        if len(next_pauo_df) < 5:
            skipped = row.drop(labels=["Time_dt"], errors="ignore").to_dict()
            skipped["Stage2_Skip_Reason"] = "后续PAUO有效点少于5个"
            skipped_rows.append(skipped)
            continue

        fit_df = pd.concat(
            [pulse_df, next_pauo_df],
            ignore_index=True,
            sort=False,
        ).sort_values("Time_dt").reset_index(drop=True)

        t_s = (
            fit_df["Time_dt"] - pulse_start_time
        ).dt.total_seconds().to_numpy(dtype=float)
        current_a = fit_df["Current"].to_numpy(dtype=float)
        voltage_v = fit_df["Voltage"].to_numpy(dtype=float)

        pulse_mask = fit_df["Zustand"].map(_is_pulse_state).to_numpy(dtype=bool)
        pauo_mask = fit_df["Zustand_clean"].eq("PAUO").to_numpy(dtype=bool)

        weights = (
            _time_support_weights(t_s, pulse_mask, PULSE_WEIGHT_TOTAL)
            + _time_support_weights(t_s, pauo_mask, PAUO_WEIGHT_TOTAL)
        )

        pulse_duration_s = max(
            float((pulse_end_time - pulse_start_time).total_seconds()), 0.0
        )
        pauo_duration_s = max(
            float(
                (
                    next_pauo_df["Time_dt"].iloc[-1]
                    - next_pauo_df["Time_dt"].iloc[0]
                ).total_seconds()
            ),
            0.0,
        )

        windows.append({
            "row": row.drop(labels=["Time_dt"], errors="ignore").to_dict(),
            "fit_df": fit_df,
            "t_s": t_s,
            "current_a": current_a,
            "voltage_v": voltage_v,
            "weights": weights,
            "pulse_mask": pulse_mask,
            "pauo_mask": pauo_mask,
            "ocv_before": float(prev_pauo_point["Voltage"]),
            "r0_ohm": r0_ohm,
            "pulse_duration_s": pulse_duration_s,
            "pauo_duration_s": pauo_duration_s,
        })

    skipped_df = pd.DataFrame(skipped_rows)
    return windows, skipped_df


def _simulate_rc_state(t_s, current_a, resistance_ohm, tau_s):
    """一阶 R||C 支路在分段常电流下的精确离散更新。"""
    t_s = np.asarray(t_s, dtype=float)
    current_a = np.asarray(current_a, dtype=float)
    voltage_state = np.zeros(len(t_s), dtype=float)

    tau_s = max(float(tau_s), 1e-12)

    for k in range(1, len(t_s)):
        dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
        current_interval = 0.5 * (current_a[k - 1] + current_a[k])
        decay = np.exp(-dt / tau_s)
        voltage_state[k] = (
            decay * voltage_state[k - 1]
            + resistance_ohm * (1.0 - decay) * current_interval
        )

    return voltage_state


def _simulate_warburg_open_state(
    t_s,
    current_a,
    rd_ohm,
    td_s,
    n_terms,
):
    """
    有限长 Warburg Open 的 Boukamp/部分分式时域级数。

    返回值包含：
      1) n=0 的积分状态：反射边界低频电容行为；
      2) n=1..N 的指数扩散模态。
    """
    t_s = np.asarray(t_s, dtype=float)
    current_a = np.asarray(current_a, dtype=float)

    td_s = max(float(td_s), 1e-12)
    rd_ohm = float(rd_ohm)
    n_terms = int(n_terms)

    voltage_w = np.zeros(len(t_s), dtype=float)
    integral_state = 0.0
    mode_states = np.zeros(n_terms, dtype=float)

    mode_index = np.arange(1, n_terms + 1, dtype=float)
    lambdas = (mode_index ** 2) * (np.pi ** 2) / td_s

    for k in range(1, len(t_s)):
        dt = max(float(t_s[k] - t_s[k - 1]), 0.0)
        current_interval = 0.5 * (current_a[k - 1] + current_a[k])

        integral_state += (rd_ohm / td_s) * current_interval * dt

        decay = np.exp(-lambdas * dt)
        mode_states = (
            decay * mode_states
            + (2.0 * rd_ohm / td_s)
            * ((1.0 - decay) / lambdas)
            * current_interval
        )

        voltage_w[k] = integral_state + mode_states.sum()

    return voltage_w


def _simulate_randles_warburg_open(
    param,
    t_s,
    current_a,
    ocv_before,
    r0_ohm,
    n_terms,
):
    """
    param = [R1_ohm, Rd_ohm, tau1_s, td_s, OCV_offset_V]
    """
    r1_ohm, rd_ohm, tau1_s, td_s, ocv_offset_v = param

    v_r1 = _simulate_rc_state(t_s, current_a, r1_ohm, tau1_s)
    v_w = _simulate_warburg_open_state(
        t_s, current_a, rd_ohm, td_s, n_terms
    )

    return (
        float(ocv_before)
        + float(ocv_offset_v)
        + current_a * float(r0_ohm)
        + v_r1
        + v_w
    )


def _warburg_initial_and_bounds(window):
    row = window["row"]
    soc_label = str(row.get("SOC", "")).strip()

    current_a = window["current_a"]
    voltage_v = window["voltage_v"]
    pulse_mask = window["pulse_mask"]
    ocv_before = window["ocv_before"]
    r0_ohm = window["r0_ohm"]

    pulse_current = np.abs(current_a[pulse_mask])
    i_pulse_abs = float(np.nanmedian(pulse_current)) if len(pulse_current) else np.nan

    pulse_voltage = voltage_v[pulse_mask]
    tail_count = min(5, len(pulse_voltage))
    v_pulse_tail = (
        float(np.nanmedian(pulse_voltage[-tail_count:]))
        if tail_count
        else np.nan
    )

    # 10% SOC 已改由 fit_rc2_coulomb_10pct 处理，此处仅覆盖 50%/90%。
    soc_key = soc_label if soc_label in RANDLES_WARBURG_INITIAL_BY_SOC else "50%"
    initial_cfg = RANDLES_WARBURG_INITIAL_BY_SOC[soc_key]
    bound_cfg = RANDLES_WARBURG_BOUNDS_BY_SOC[soc_key]

    if (
        np.isfinite(i_pulse_abs)
        and i_pulse_abs > ZERO_CURRENT_LIMIT
        and np.isfinite(v_pulse_tail)
    ):
        r_dyn_guess = abs(v_pulse_tail - ocv_before) / i_pulse_abs - r0_ohm
    else:
        r_dyn_guess = 0.02

    r_dyn_guess = float(np.clip(r_dyn_guess, 0.002, 0.20))
    r1_fraction = float(
        np.clip(initial_cfg["R1_fraction"], 0.05, 0.95)
    )

    x0 = np.array([
        r1_fraction * r_dyn_guess,
        (1.0 - r1_fraction) * r_dyn_guess,
        float(initial_cfg["tau1"]),
        float(initial_cfg["td"]),
        0.0,
    ], dtype=float)

    lower = np.array([
        1e-8,
        1e-8,
        float(bound_cfg["tau1_min"]),
        float(bound_cfg["td_min"]),
        -0.030,
    ], dtype=float)

    upper = np.array([
        float(bound_cfg["R1_max"]),
        float(bound_cfg["Rd_max"]),
        float(bound_cfg["tau1_max"]),
        float(bound_cfg["td_max"]),
        0.030,
    ], dtype=float)

    x0 = np.clip(x0, lower + 1e-12, upper - 1e-12)
    return x0, lower, upper


def _fit_metrics(voltage_true, voltage_pred):
    error_mv = (np.asarray(voltage_pred) - np.asarray(voltage_true)) * 1000.0
    rmse_mv = float(np.sqrt(np.mean(error_mv ** 2)))
    mae_mv = float(np.mean(np.abs(error_mv)))

    ss_res = float(np.sum((voltage_true - voltage_pred) ** 2))
    ss_tot = float(np.sum((voltage_true - np.mean(voltage_true)) ** 2))
    r2_score = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return rmse_mv, mae_mv, r2_score


def _bound_flags(fit_x, lower, upper, names):
    fit_x = np.asarray(fit_x, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)

    span = np.maximum(upper - lower, 1e-12)
    tolerance = 1e-4 * span + 1e-10

    flags = []
    for value, lo, hi, tol, name in zip(
        fit_x, lower, upper, tolerance, names
    ):
        if value <= lo + tol:
            flags.append(f"{name}:lower")
        if value >= hi - tol:
            flags.append(f"{name}:upper")

    return "；".join(flags) if flags else "无"


def fit_randles_warburg_open(windows, n_terms=5):
    result_rows = []

    for window in windows:
        row = window["row"]
        t_s = window["t_s"]
        current_a = window["current_a"]
        voltage_v = window["voltage_v"]
        weights = window["weights"]
        ocv_before = window["ocv_before"]
        r0_ohm = window["r0_ohm"]

        x0, lower, upper = _warburg_initial_and_bounds(window)

        def residual(param):
            voltage_hat = _simulate_randles_warburg_open(
                param,
                t_s,
                current_a,
                ocv_before,
                r0_ohm,
                n_terms,
            )

            tau1_s = float(param[2])
            td_s = float(param[3])
            separation_violation = max(0.0, 3.0 * tau1_s - td_s)
            separation_penalty_v = (
                0.002 * separation_violation / max(3.0 * tau1_s, 1e-9)
            )

            return np.concatenate([
                (voltage_hat - voltage_v) * weights,
                np.array([separation_penalty_v], dtype=float),
            ])

        output_row = dict(row)

        try:
            fit = least_squares(
                residual,
                x0=x0,
                bounds=(lower, upper),
                max_nfev=8000,
                x_scale="jac",
                loss="soft_l1",
                f_scale=5e-4,
            )

            r1_ohm, rd_ohm, tau1_s, td_s, ocv_offset_v = fit.x
            voltage_fit = _simulate_randles_warburg_open(
                fit.x,
                t_s,
                current_a,
                ocv_before,
                r0_ohm,
                n_terms,
            )

            rmse_mv, mae_mv, r2_score = _fit_metrics(
                voltage_v, voltage_fit
            )

            if int(n_terms) >= 5:
                voltage_fit_n3_same_param = _simulate_randles_warburg_open(
                    fit.x,
                    t_s,
                    current_a,
                    ocv_before,
                    r0_ohm,
                    WARBURG_CHECK_TERMS,
                )
                truncation_error_mv = (
                    voltage_fit_n3_same_param - voltage_fit
                ) * 1000.0
                n3_n5_trunc_rmse_mv = float(
                    np.sqrt(np.mean(truncation_error_mv ** 2))
                )
                n3_n5_trunc_max_mv = float(
                    np.max(np.abs(truncation_error_mv))
                )
            else:
                n3_n5_trunc_rmse_mv = np.nan
                n3_n5_trunc_max_mv = np.nan

            c1_f = tau1_s / r1_ohm if r1_ohm > 0 else np.nan

            output_row.update({
                "R1": r1_ohm * 1000.0,
                "R2": rd_ohm * 1000.0,
                "tau1": tau1_s,
                "tau2": td_s,
                "C1": c1_f,
                "C2": np.nan,
                "Rd_internal_mOhm": rd_ohm * 1000.0,
                "td_internal_s": td_s,
                "OCV_offset_mV": ocv_offset_v * 1000.0,
                "RMSE_mV": rmse_mv,
                "MAE_mV": mae_mv,
                "R2_score": r2_score,
                "Fit_Success": bool(fit.success),
                "Fit_Status": int(fit.status),
                "Fit_Message": str(fit.message),
                "Fit_NFEV": int(fit.nfev),
                "Fit_Boundary_Flags": _bound_flags(
                    fit.x,
                    lower,
                    upper,
                    ["R1", "Rd", "tau1", "td", "OCV_offset"],
                ),
                "Warburg_N_Terms": int(n_terms),
                "Warburg_N3_vs_N5_Trunc_RMSE_mV": n3_n5_trunc_rmse_mv,
                "Warburg_N3_vs_N5_Trunc_Max_mV": n3_n5_trunc_max_mv,
                "Truncation_Below_Configured_Noise": (
                    bool(n3_n5_trunc_rmse_mv <= WARBURG_NOISE_FLOOR_MV)
                    if np.isfinite(n3_n5_trunc_rmse_mv)
                    else np.nan
                ),
                "Pulse_Window_s": window["pulse_duration_s"],
                "PAUO_Window_s": window["pauo_duration_s"],
                "Fit_Point_Count": len(t_s),
                "R2_Definition": (
                    "R2 = Rd of finite-length reflective Warburg Open"
                ),
                "tau2_Definition": (
                    "tau2 = td, Warburg characteristic diffusion time"
                ),
                "C2_Definition": (
                    "Not applicable after replacing R2||C2 with Warburg Open"
                ),
                "Model_Topology": "R0 + (R1||C1) + Warburg Open",
            })

        except Exception as exc:
            output_row.update({
                "R1": np.nan,
                "R2": np.nan,
                "tau1": np.nan,
                "tau2": np.nan,
                "C1": np.nan,
                "C2": np.nan,
                "Rd_internal_mOhm": np.nan,
                "td_internal_s": np.nan,
                "OCV_offset_mV": np.nan,
                "RMSE_mV": np.nan,
                "MAE_mV": np.nan,
                "R2_score": np.nan,
                "Fit_Success": False,
                "Fit_Status": np.nan,
                "Fit_Message": f"{type(exc).__name__}: {exc}",
                "Fit_NFEV": np.nan,
                "Fit_Boundary_Flags": "拟合异常",
                "Warburg_N_Terms": int(n_terms),
                "Warburg_N3_vs_N5_Trunc_RMSE_mV": np.nan,
                "Warburg_N3_vs_N5_Trunc_Max_mV": np.nan,
                "Truncation_Below_Configured_Noise": np.nan,
                "Pulse_Window_s": window["pulse_duration_s"],
                "PAUO_Window_s": window["pauo_duration_s"],
                "Fit_Point_Count": len(t_s),
                "R2_Definition": (
                    "R2 = Rd of finite-length reflective Warburg Open"
                ),
                "tau2_Definition": (
                    "tau2 = td, Warburg characteristic diffusion time"
                ),
                "C2_Definition": (
                    "Not applicable after replacing R2||C2 with Warburg Open"
                ),
                "Model_Topology": "R0 + (R1||C1) + Warburg Open",
            })

        result_rows.append(output_row)

    return pd.DataFrame(result_rows)


def _rc2_coulomb_10pct_sub_key(row, i_pulse_abs):
    """10% SOC 子工况分类：实验中只有 CHA_1.5A / CHA_3A / DCH_3A 三种。"""
    zustand = str(row.get("Zustand", "")).strip()
    current_tag = "1.5A" if (
        np.isfinite(i_pulse_abs) and i_pulse_abs < 2.25
    ) else "3A"
    prefix = "CHA" if zustand.startswith("CHA") else "DCH"
    sub_key = f"{prefix}_{current_tag}"
    if sub_key not in RC2_COULOMB_10PCT_CONFIG:
        sub_key = "CHA_3A"
    return sub_key


def fit_rc2_coulomb_10pct(windows):
    """
    10% SOC 专用模型：
        V(t) = OCV_before + offset + I*R0 + V_RC1 + V_RC2 + k * Q(t)

    其中 Q(t) 为脉冲起始至 t 的累计电荷（梯形积分），
    k = dOCV/dQ，承接 10% SOC 陡峭 OCV 区脉冲期间的 OCV 移动；
    弛豫期间 Q 恒定，k*Q 退化为常数，弛豫形状仍由两个 RC 支路描述。
    输出列与 fit_randles_warburg_open 对齐（R1/R2/tau1/tau2/C1/C2 等），
    并额外给出 k_OCV_mV_per_C。
    """
    shared = RC2_COULOMB_10PCT_SHARED
    result_rows = []

    for window in windows:
        row = window["row"]
        t_s = window["t_s"]
        current_a = window["current_a"]
        voltage_v = window["voltage_v"]
        weights = window["weights"]
        ocv_before = window["ocv_before"]
        r0_ohm = window["r0_ohm"]
        pauo_duration_s = float(window.get("pauo_duration_s", np.nan))

        pulse_mask = window["pulse_mask"]
        pulse_current = np.abs(np.asarray(current_a)[np.asarray(pulse_mask)])
        i_pulse_abs = (
            float(np.nanmedian(pulse_current)) if len(pulse_current) else np.nan
        )
        sub_key = _rc2_coulomb_10pct_sub_key(row, i_pulse_abs)
        sub_cfg = RC2_COULOMB_10PCT_CONFIG[sub_key]

        t_arr = np.asarray(t_s, dtype=float)
        i_arr = np.asarray(current_a, dtype=float)
        charge_c = np.concatenate([
            [0.0],
            np.cumsum(0.5 * (i_arr[1:] + i_arr[:-1]) * np.diff(t_arr)),
        ])

        tau2_upper = float(shared["tau2_max"])
        if np.isfinite(pauo_duration_s) and pauo_duration_s > 0:
            tau2_upper = min(tau2_upper, 0.9 * pauo_duration_s)
        tau2_upper = max(tau2_upper, float(shared["tau2_min"]) + 1.0)

        def simulate(param):
            r1_ohm, r2_ohm, tau1_s, tau2_s, k_v_per_c, offset_v = param
            return (
                float(ocv_before)
                + float(offset_v)
                + i_arr * float(r0_ohm)
                + _simulate_rc_state(t_arr, i_arr, r1_ohm, tau1_s)
                + _simulate_rc_state(t_arr, i_arr, r2_ohm, tau2_s)
                + float(k_v_per_c) * charge_c
            )

        def residual(param):
            return (simulate(param) - voltage_v) * weights

        x0 = np.array([
            float(shared["R1_init"]),
            float(shared["R2_init"]),
            float(shared["tau1_init"]),
            float(sub_cfg["tau2_init"]),
            float(shared["k_init"]),
            0.0,
        ], dtype=float)
        lower = np.array([
            1e-6,
            1e-6,
            float(shared["tau1_min"]),
            float(shared["tau2_min"]),
            float(shared["k_min"]),
            -float(shared["ocv_offset_slack_v"]),
        ], dtype=float)
        upper = np.array([
            float(shared["R1_max"]),
            float(sub_cfg["R2_max"]),
            float(shared["tau1_max"]),
            tau2_upper,
            float(shared["k_max"]),
            float(shared["ocv_offset_slack_v"]),
        ], dtype=float)
        x0 = np.clip(x0, lower + 1e-12, upper - 1e-12)

        output_row = dict(row)

        try:
            fit = least_squares(
                residual,
                x0=x0,
                bounds=(lower, upper),
                max_nfev=10000,
                x_scale="jac",
                loss="soft_l1",
                f_scale=5e-4,
            )

            r1_ohm, r2_ohm, tau1_s, tau2_s, k_v_per_c, offset_v = fit.x
            voltage_fit = simulate(fit.x)
            rmse_mv, mae_mv, r2_score = _fit_metrics(voltage_v, voltage_fit)

            output_row.update({
                "R1": r1_ohm * 1000.0,
                "R2": r2_ohm * 1000.0,
                "tau1": tau1_s,
                "tau2": tau2_s,
                "C1": tau1_s / r1_ohm if r1_ohm > 0 else np.nan,
                "C2": tau2_s / r2_ohm if r2_ohm > 0 else np.nan,
                "k_OCV_mV_per_C": k_v_per_c * 1000.0,
                "OCV_offset_mV": offset_v * 1000.0,
                "RMSE_mV": rmse_mv,
                "MAE_mV": mae_mv,
                "R2_score": r2_score,
                "Fit_Success": bool(fit.success),
                "Fit_Status": int(fit.status),
                "Fit_Message": str(fit.message),
                "Fit_NFEV": int(fit.nfev),
                "Fit_Boundary_Flags": _bound_flags(
                    fit.x,
                    lower,
                    upper,
                    ["R1", "R2", "tau1", "tau2", "k_OCV", "OCV_offset"],
                ),
                "RC2_Coulomb_Sub_Key": sub_key,
                "R2_Definition": (
                    "R2 = slow RC branch resistance "
                    "(10% SOC: R0 + R1||C1 + R2||C2 + k*Q model)"
                ),
                "tau2_Definition": "tau2 = slow RC branch time constant",
                "C2_Definition": "C2 = tau2 / R2 of slow RC branch",
                "Model_Topology": "R0 + (R1||C1) + (R2||C2) + k*Q OCV drift",
            })

        except Exception as exc:
            output_row.update({
                "R1": np.nan, "R2": np.nan,
                "tau1": np.nan, "tau2": np.nan,
                "C1": np.nan, "C2": np.nan,
                "k_OCV_mV_per_C": np.nan,
                "OCV_offset_mV": np.nan,
                "RMSE_mV": np.nan, "MAE_mV": np.nan, "R2_score": np.nan,
                "Fit_Success": False,
                "Fit_Status": np.nan,
                "Fit_Message": f"{type(exc).__name__}: {exc}",
                "Fit_NFEV": np.nan,
                "Fit_Boundary_Flags": "拟合异常",
                "RC2_Coulomb_Sub_Key": sub_key,
                "R2_Definition": (
                    "R2 = slow RC branch resistance "
                    "(10% SOC: R0 + R1||C1 + R2||C2 + k*Q model)"
                ),
                "tau2_Definition": "tau2 = slow RC branch time constant",
                "C2_Definition": "C2 = tau2 / R2 of slow RC branch",
                "Model_Topology": "R0 + (R1||C1) + (R2||C2) + k*Q OCV drift",
            })

        output_row.update({
            "Pulse_Window_s": window["pulse_duration_s"],
            "PAUO_Window_s": window["pauo_duration_s"],
            "Fit_Point_Count": len(t_s),
        })
        result_rows.append(output_row)

    return pd.DataFrame(result_rows)


In [9]:
# =============================================================================
# 13. 主拟合：50%/90% SOC 用 N=5 Warburg Open；10% SOC 用 RC2+Coulomb
# =============================================================================
stage2_windows, stage2_skipped = build_stage2_fit_windows(
    time_diff_sequence,
    r0_result,
)

stage2_windows_10pct = [
    w for w in stage2_windows
    if str(w["row"].get("SOC", "")).strip() == "10%"
]
stage2_windows_other = [
    w for w in stage2_windows
    if str(w["row"].get("SOC", "")).strip() != "10%"
]

r1r2_result_warburg = fit_randles_warburg_open(
    stage2_windows_other,
    n_terms=WARBURG_MAIN_TERMS,
)
r1r2_result_rc2k = fit_rc2_coulomb_10pct(stage2_windows_10pct)

r1r2_result = pd.concat(
    [r1r2_result_warburg, r1r2_result_rc2k],
    ignore_index=True,
    sort=False,
)

print(
    f"Stage 2 可拟合窗口: {len(stage2_windows)}"
    f"（50%/90% Warburg: {len(stage2_windows_other)}；"
    f"10% RC2+Coulomb: {len(stage2_windows_10pct)}）；"
    f"跳过记录: {len(stage2_skipped)}"
)
print(
    "R2/tau2 含义按 SOC 区分："
    "50%/90% 为 Warburg 参数（R2=Rd, tau2=td, C2 不适用）；"
    "10% 为慢 RC 支路参数（R2/tau2/C2 恢复 RC 含义，"
    "另有 k_OCV_mV_per_C 表示 OCV 库仑斜率）。"
    "详见每行的 Model_Topology 与 *_Definition 列。"
)

main_display_columns = [
    "SOH",
    "SOC",
    "Time",
    "Zustand",
    "Current",
    "R0_mOhm",
    "R1",
    "R2",
    "tau1",
    "tau2",
    "C1",
    "C2",
    "k_OCV_mV_per_C",
    "RMSE_mV",
    "R2_score",
    "Fit_Boundary_Flags",
    "Model_Topology",
    "Warburg_N_Terms",
    "Warburg_N3_vs_N5_Trunc_RMSE_mV",
    "Truncation_Below_Configured_Noise",
    "File",
]

main_display_columns = [
    col for col in main_display_columns
    if col in r1r2_result.columns
]

display(
    r1r2_result[main_display_columns]
    .sort_values(
        ["SOC", "Zustand", "Current", "SOH"],
        ascending=[True, True, True, False],
    )
    .reset_index(drop=True)
)

warburg_summary = (
    r1r2_result
    .groupby(["SOC", "Model_Topology"], as_index=False)
    .agg(
        Segment_Count=("R2", "size"),
        Fit_Success_Count=("Fit_Success", "sum"),
        Mean_RMSE_mV=("RMSE_mV", "mean"),
        Median_RMSE_mV=("RMSE_mV", "median"),
        Mean_R2_score=("R2_score", "mean"),
        Mean_R2_mOhm=("R2", "mean"),
        Mean_tau2_s=("tau2", "mean"),
        Mean_N3_N5_Trunc_RMSE_mV=(
            "Warburg_N3_vs_N5_Trunc_RMSE_mV",
            "mean",
        ),
    )
)

display(warburg_summary)

if not stage2_skipped.empty:
    print("Stage 2 跳过记录及原因：")
    display(stage2_skipped)


Stage 2 可拟合窗口: 203；跳过记录: 13
当前输出中的 R2/tau2 已改为 Warburg 参数：R2=Rd，tau2=td；C2 不再适用。


,SOH,SOC,Time,Zustand,Current,R0_mOhm,R1,R2,tau1,tau2,C1,C2,RMSE_mV,R2_score,Fit_Boundary_Flags,Warburg_N_Terms,Warburg_N3_vs_N5_Trunc_RMSE_mV,Truncation_Below_Configured_Noise,File
0,81.2,10%,2025-07-14 07:38:42.150000+00:00,CHA,1.496636,24.561924,7.852202,25.368423,11.321484,30.926185,1.441823e+03,NaN,0.769692,0.989862,无,5,0.142878,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM36_...
1,96.2,10%,2024-09-06 06:45:06.580000+00:00,CHA,1.496906,18.759214,0.000010,599.978251,90.000000,8616.339160,9.000000e+09,NaN,3.520453,0.316582,R1:lower；Rd:upper；tau1:upper,5,0.929336,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM4_9...
2,83.9,10%,2025-03-20 03:01:56.710000+00:00,CHA,1.496996,23.451344,0.000010,599.966559,0.203190,7884.916752,2.031896e+07,NaN,3.618371,0.733251,R1:lower；Rd:upper；tau1:lower,5,1.109519,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM26_...
3,81.7,10%,2025-06-27 01:42:05.290000+00:00,CHA,1.498165,25.229957,0.000010,599.141737,0.683487,7369.115782,6.834863e+07,NaN,3.674318,0.839895,R1:lower,5,1.404462,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...
4,78.6,10%,2025-11-10 11:28:15.940000+00:00,CHA,1.498165,26.634383,8.616684,27.295772,12.598810,34.592539,1.462141e+03,NaN,0.772514,0.991681,无,5,0.154742,True,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
198,77.5,90%,2026-01-04 13:05:24.440000+00:00,DCH,-2.991947,28.004247,23.508100,111.591736,0.200000,969.800257,8.507706e+00,NaN,1.052122,0.998728,tau1:lower,5,0.980521,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM54_...
199,77.8,90%,2025-12-14 14:48:59.310000+00:00,DCH,-2.991407,32.327523,18.678278,111.915579,0.200000,990.926002,1.070763e+01,NaN,1.095422,0.998727,tau1:lower,5,1.025636,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM52_...
200,81.2,90%,2025-07-13 17:17:06.590000+00:00,DCH,-2.991047,33.025694,10.979068,92.301928,0.215459,764.012144,1.962457e+01,NaN,0.876029,0.998853,无,5,0.860787,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM36_...
201,78.6,90%,2025-11-09 21:13:10.940000+00:00,DCH,-2.991047,32.660834,16.862174,108.861165,0.200000,963.940755,1.186087e+01,NaN,1.020728,0.998836,tau1:lower,5,1.004167,False,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...


,SOC,Segment_Count,Fit_Success_Count,Mean_RMSE_mV,Median_RMSE_mV,Mean_R2_score,Mean_R2_mOhm,Mean_tau2_s,Mean_N3_N5_Trunc_RMSE_mV
0,10%,72,72,1.966885,2.367976,0.925029,143.617211,1689.202612,0.420856
1,50%,71,71,0.236490,0.219069,0.999621,101.982328,1805.272086,0.647449
2,90%,60,60,0.518240,0.471490,0.999229,84.470313,616.947867,0.635784


Stage 2 跳过记录及原因：


,SOH,SOC,File,Time,Current,Voltage,Zustand,ID,Zustand/Current,R0,R0_Target_Time,Pause_to_Pulse_Time_Diff_s,R0_Quality,R0_Confidence,R0_mOhm,Current_abs_A,Current_Label,SOC_pulse_current,R0_Is_Valid,Stage2_Skip_Reason
0,82.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM32_...,2025-06-01 04:50:52.090000+00:00,2.994096,4.198187,CHA,32_29,CHA/3.0,NaN,2025-06-01 04:50:51.660000+00:00,0.83,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,3.0,3.0A,90% CHA 3.0A,False,R0无效
1,81.7,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...,2025-06-26 12:20:22.870000+00:00,2.370311,4.199966,CHA,34_29,CHA/2.4,NaN,2025-06-26 12:20:04.430000+00:00,18.80,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,NaN,2.4,2.4A,90% CHA 2.4A,False,R0无效
2,81.7,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM34_...,2025-06-26 20:02:05.060000+00:00,2.999851,3.879424,CHA,34_47,CHA/3.0,NaN,2025-06-26 20:01:53.770000+00:00,11.66,pause结束点到pulse首点时间差>5s,不可用（权重0.0）,NaN,3.0,3.0A,50% CHA 3.0A,False,R0无效
3,81.2,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM36_...,2025-07-13 18:18:10.370000+00:00,2.993196,4.198521,CHA,36_30,CHA/3.0,NaN,2025-07-13 18:18:09.940000+00:00,0.82,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,3.0,3.0A,90% CHA 3.0A,False,R0无效
4,80.8,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM38_...,2025-07-31 14:28:35.310000+00:00,2.881318,4.200634,CHA,38_29,CHA/2.9,NaN,2025-07-31 14:28:34.860000+00:00,0.84,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,2.9,2.9A,90% CHA 2.9A,False,R0无效
5,80.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM40_...,2025-08-17 20:01:15+00:00,2.993196,4.197187,CHA,40_29,CHA/3.0,NaN,2025-08-17 20:01:14.580000+00:00,0.82,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,3.0,3.0A,90% CHA 3.0A,False,R0无效
6,79.8,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM42_...,2025-09-17 11:59:52.180000+00:00,2.992657,4.196964,CHA,42_29,CHA/3.0,NaN,2025-09-17 11:59:51.770000+00:00,0.81,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,3.0,3.0A,90% CHA 3.0A,False,R0无效
7,79.4,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM44_...,2025-10-04 17:07:30.340000+00:00,2.597305,4.200300,CHA,44_29,CHA/2.6,NaN,2025-10-04 17:07:29.990000+00:00,0.75,首点0且电压跳变；无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,2.6,2.6A,90% CHA 2.6A,False,R0无效
8,79.0,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM46_...,2025-10-23 17:27:14.830000+00:00,2.985642,4.193740,CHA,46_29,CHA/3.0,NaN,2025-10-23 17:27:14.450000+00:00,0.79,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,3.0,3.0A,90% CHA 3.0A,False,R0无效
9,78.6,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM48_...,2025-11-09 22:14:14.040000+00:00,2.967295,4.199855,CHA,48_29,CHA/3.0,NaN,2025-11-09 22:14:13.710000+00:00,0.76,无法计算R0：早期拟合窗口电流不稳定,不可用（权重0.0）,NaN,3.0,3.0A,90% CHA 3.0A,False,R0无效
